# TSP DRL — Independent validation and final evaluation (v1.1)

**Purpose:** continue the completed `clean_restart_seed20261013_v1` pilot without retraining. This notebook selects a checkpoint on a *new validation set* at $n=20$, then evaluates the frozen checkpoint and fixed classical baselines on a **separate, newly generated final test set** at $n\in\{20,50,100\}$.

## Scientific safeguards

- **No training cell.** Existing checkpoint files are read-only inputs.
- The original pilot folder and original test instances (seed 4202026) are never overwritten.
- Validation seed: `7302026`; final-test seed: `8302026`; both differ from the pilot's old test seed.
- Selection rule is predeclared: choose the available checkpoint with the lowest mean validation tour length at $n=20$ under **greedy decoding** only. Do not select based on the final-test results.
- Final test: evaluate the selected checkpoint with greedy decoding and best-of-512 sampling, plus NN, NN + 2-opt, and fixed-parameter ACO, on the same 100 fresh instances for each size.
- A reported gap is against the best observed method on that instance, **not** an exact optimality gap.
- Timings use a separate warm-up and repeated protocol; model loading is excluded. DRL timing includes input tensor transfer, forward pass, candidate scoring, and result retrieval. CPU baseline timing includes solver execution and tour-length calculation. This remains an implementation-specific runtime comparison, not hardware-independent latency.

**Do not edit the seeds or selection rule after inspecting results.** Keep all outputs under the new `final_evaluation_v1_1` folder. If the saved pilot is not on Drive, the setup cell can restore the supplied audit ZIP without retraining.

## Step 0 — Mount Drive and locate the existing pilot

Choose **Runtime → Change runtime type → GPU** first. Run the cells in order. The pilot directory is treated as immutable.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, shutil, subprocess, hashlib, time, zipfile
import numpy as np
import pandas as pd
import torch

try:
    from google.colab import drive, files
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab before running it.") from exc

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/PFE_TSP_DRL')
RUN_ID = 'clean_restart_seed20261013_v1'
RUN_ROOT = PROJECT_ROOT / 'runs' / RUN_ID
FINAL_ROOT = PROJECT_ROOT / 'final_evaluation_v1_1'
FINAL_ROOT.mkdir(parents=True, exist_ok=True)

VALIDATION_SEED = 7302026
FINAL_TEST_SEED = 8302026
N_VALIDATION = 100
N_FINAL_TEST = 100
CITY_SIZES = [20, 50, 100]
CHECKPOINT_SUFFIXES_TO_CONSIDER = [1000, 5000, 10000, 12999]
SAMPLE_BUDGET = 512
TIMING_INSTANCES_PER_SIZE = 10
TIMING_REPEATS = 5
BOOTSTRAP_REPLICATES = 10000

print('Project root:', PROJECT_ROOT)
print('Pilot run:', RUN_ROOT)
print('New evaluation outputs:', FINAL_ROOT)
print('Python:', sys.version.replace('\n', ' '))
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. Select Runtime → Change runtime type → GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

checkpoint_dir = RUN_ROOT / 'full_train_n20' / 'Pt'
if not (RUN_ROOT / 'manifest_pre.json').is_file() or not checkpoint_dir.is_dir():
    print('\nPilot artifacts were not found in the expected Drive path.')
    print('Upload TSP_DRL_Audit_clean_restart_seed20261013_v1.zip from your computer when prompted.')
    uploaded = files.upload()
    zip_candidates = []
    for name, payload in uploaded.items():
        candidate = Path('/content') / Path(name).name
        candidate.write_bytes(payload)
        if zipfile.is_zipfile(candidate):
            with zipfile.ZipFile(candidate) as zf:
                names = zf.namelist()
                if any(n.endswith('manifest_pre.json') and RUN_ID in n for n in names) and any(n.endswith('_step12999_act.pt') for n in names):
                    zip_candidates.append(candidate)
    if len(zip_candidates) != 1:
        raise RuntimeError('Could not identify exactly one valid clean-restart ZIP. No archive was extracted.')
    if RUN_ROOT.exists():
        raise RuntimeError(f'Partial pilot directory already exists at {RUN_ROOT}; refusing to merge/overwrite. Inspect it manually first.')
    safe_tmp = Path('/content/pfe_pilot_restore')
    if safe_tmp.exists():
        shutil.rmtree(safe_tmp)
    safe_tmp.mkdir(parents=True)
    with zipfile.ZipFile(zip_candidates[0]) as zf:
        for info in zf.infolist():
            target = (safe_tmp / info.filename).resolve()
            if safe_tmp.resolve() not in target.parents and target != safe_tmp.resolve():
                raise RuntimeError(f'Unsafe archive path rejected: {info.filename}')
        zf.extractall(safe_tmp)
    extracted_root = safe_tmp / RUN_ID
    if not (extracted_root / 'manifest_pre.json').is_file() or not (extracted_root / 'full_train_n20' / 'Pt').is_dir():
        raise RuntimeError('Archive extracted but expected pilot files are missing; inspect the ZIP layout.')
    RUN_ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(extracted_root, RUN_ROOT)
    checkpoint_dir = RUN_ROOT / 'full_train_n20' / 'Pt'
    print('Pilot artifacts restored to Drive. No training was run.')

manifest_pre_path = RUN_ROOT / 'manifest_pre.json'
training_manifest_path = RUN_ROOT / 'manifest_training.json'
if not manifest_pre_path.is_file() or not training_manifest_path.is_file():
    raise FileNotFoundError('The pilot manifest files are incomplete. Do not train again; inspect the saved ZIP.')
manifest_pre = json.loads(manifest_pre_path.read_text())
training_manifest = json.loads(training_manifest_path.read_text())
print('\nPilot commit:', manifest_pre.get('commit_sha'))
print('Configured updates:', training_manifest.get('training_steps_configured'))
print('Existing checkpoints:', len(list(checkpoint_dir.glob('*_step*_act.pt'))))
print('Existing original test seed (must not be reused):', manifest_pre.get('test_seed'))

## Step 1 — Load the exact source revision and define the evaluator

The actor implementation is checked out at the commit recorded in the pilot manifest. This cell does not install packages or change the saved pilot.

In [ ]:
REPO_URL = manifest_pre.get('repository_url', 'https://github.com/Rintarooo/TSP_DRL_PtrNet.git')
COMMIT_SHA = manifest_pre.get('commit_sha')
if not COMMIT_SHA:
    raise ValueError('commit_sha is missing from manifest_pre.json.')
REPO_DIR = Path('/content/TSP_DRL_PtrNet_eval')
if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', COMMIT_SHA], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != COMMIT_SHA:
    raise RuntimeError(f'Source commit mismatch: expected {COMMIT_SHA}, got {actual_commit}')
for required in ['actor.py', 'env.py']:
    if not (REPO_DIR / required).is_file():
        raise FileNotFoundError(f'Missing required source file: {required}')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from types import SimpleNamespace
from actor import PtrNet1

DEVICE = torch.device('cuda:0')
MODEL_ARGS = {'embed': 128, 'hidden': 128, 'init_min': -0.08, 'init_max': 0.08,
              'clip_logits': 10, 'softmax_T': 1.0, 'n_glimpse': 1}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def parse_suffix(path):
    import re
    m = re.search(r'_step(\d+)_act\.pt$', Path(path).name)
    return int(m.group(1)) if m else None

def completed_updates_from_suffix(suffix):
    # The upstream final-save filename uses zero-based loop index 12999 after 13,000 updates.
    return suffix + 1 if suffix == 12999 else suffix

def load_actor(checkpoint_path, decode_type):
    cfg = SimpleNamespace(**MODEL_ARGS, decode_type=decode_type)
    model = PtrNet1(cfg).to(DEVICE)
    try:
        state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model

def tour_length_np(coords, tour):
    coords = np.asarray(coords, dtype=np.float64)
    tour = np.asarray(tour, dtype=np.int64)
    ordered = coords[tour]
    return float(np.linalg.norm(ordered[1:] - ordered[:-1], axis=1).sum()
                 + np.linalg.norm(ordered[0] - ordered[-1]))

def predict_one_instance(model, coords, sample_count, inference_seed):
    torch.manual_seed(int(inference_seed))
    torch.cuda.manual_seed_all(int(inference_seed))
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    x0 = torch.as_tensor(coords, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    x = x0.repeat(sample_count, 1, 1)
    with torch.inference_mode():
        tours, _ = model(x, DEVICE)
        ordered = torch.gather(x, 1, tours.long().unsqueeze(-1).expand(-1, -1, 2))
        lengths = (torch.linalg.vector_norm(ordered[:, 1:] - ordered[:, :-1], dim=-1).sum(dim=1)
                   + torch.linalg.vector_norm(ordered[:, 0] - ordered[:, -1], dim=-1))
        best_idx = torch.argmin(lengths)
        best_tour = tours[best_idx].detach().cpu().numpy().astype(np.int64)
        model_length = float(lengths[best_idx].item())
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    if sorted(best_tour.tolist()) != list(range(len(coords))):
        raise AssertionError('Actor returned a non-permutation tour.')
    cpu_length = tour_length_np(coords, best_tour)
    if abs(cpu_length - model_length) > 5e-5:
        raise AssertionError(f'Tour length mismatch: GPU={model_length}, CPU={cpu_length}')
    return best_tour, cpu_length, elapsed

# Fixed classical baselines: same definitions and parameters as the pilot.
def distance_matrix(coords):
    delta = np.asarray(coords)[:, None, :] - np.asarray(coords)[None, :, :]
    return np.sqrt(np.sum(delta * delta, axis=-1))

def nearest_neighbor(coords, start=0):
    coords = np.asarray(coords, dtype=np.float64); n = len(coords); d = distance_matrix(coords)
    tour = [int(start)]; unvisited = set(range(n)); unvisited.remove(int(start))
    while unvisited:
        current = tour[-1]
        nxt = min(unvisited, key=lambda j: (d[current, j], j))
        tour.append(nxt); unvisited.remove(nxt)
    return np.asarray(tour, dtype=np.int64)

def two_opt(coords, initial_tour, max_passes=100):
    coords = np.asarray(coords, dtype=np.float64); d = distance_matrix(coords)
    tour = np.asarray(initial_tour, dtype=np.int64).copy(); n = len(tour)
    for _ in range(max_passes):
        improved = False
        for i in range(n - 1):
            for j in range(i + 2, n):
                if i == 0 and j == n - 1: continue
                a, b = tour[i], tour[(i + 1) % n]; c, e = tour[j], tour[(j + 1) % n]
                if d[a, b] + d[c, e] > d[a, c] + d[b, e] + 1e-12:
                    tour[i + 1:j + 1] = tour[i + 1:j + 1][::-1]
                    improved = True; break
            if improved: break
        if not improved: break
    return tour

ACO_ANTS, ACO_ITERATIONS, ACO_ALPHA, ACO_BETA, ACO_RHO, ACO_Q = 15, 30, 1.0, 3.0, 0.1, 1.0
def ant_colony_tsp(coords, seed, n_ants=ACO_ANTS, n_iter=ACO_ITERATIONS,
                   alpha=ACO_ALPHA, beta=ACO_BETA, rho=ACO_RHO, q=ACO_Q):
    coords = np.asarray(coords, dtype=np.float64); n = len(coords); d = distance_matrix(coords)
    heuristic = 1.0 / np.maximum(d, 1e-12); np.fill_diagonal(heuristic, 0.0)
    pheromone = np.ones((n, n), dtype=np.float64); rng = np.random.default_rng(seed)
    best_tour, best_length = None, float('inf')
    for _ in range(n_iter):
        iteration_tours, iteration_lengths = [], []
        for _ant in range(n_ants):
            start = int(rng.integers(0, n)); tour = [start]
            unvisited = np.ones(n, dtype=bool); unvisited[start] = False
            while unvisited.any():
                current = tour[-1]; candidates = np.flatnonzero(unvisited)
                weights = np.power(pheromone[current, candidates], alpha) * np.power(heuristic[current, candidates], beta)
                weights = np.nan_to_num(weights, nan=0.0, posinf=0.0, neginf=0.0)
                if weights.sum() <= 0: nxt = int(rng.choice(candidates))
                else: nxt = int(rng.choice(candidates, p=weights / weights.sum()))
                tour.append(nxt); unvisited[nxt] = False
            tour = np.asarray(tour, dtype=np.int64); length = tour_length_np(coords, tour)
            iteration_tours.append(tour); iteration_lengths.append(length)
            if length < best_length: best_tour, best_length = tour.copy(), length
        pheromone *= (1.0 - rho)
        for tour, length in zip(iteration_tours, iteration_lengths):
            deposit = q / max(length, 1e-12)
            for k in range(n):
                a, b = int(tour[k]), int(tour[(k + 1) % n])
                pheromone[a, b] += deposit; pheromone[b, a] += deposit
    return best_tour

print('Source commit verified:', actual_commit)
print('Evaluator ready; device:', DEVICE)
print('Checkpoint directory:', checkpoint_dir)
print('Output directory:', FINAL_ROOT)

## Step 2 — Select one checkpoint using validation only (n=20, greedy)

This cell generates or reloads a fixed independent validation set. It evaluates the available 1,000 / 5,000 / 10,000 / final checkpoints with greedy decoding. The lowest mean validation tour length wins; final-test data is not accessed in this selection step.

In [ ]:
validation_dir = FINAL_ROOT / 'validation'
validation_dir.mkdir(parents=True, exist_ok=True)
validation_npz = validation_dir / f'validation_n20_seed{VALIDATION_SEED}_count{N_VALIDATION}.npz'
if validation_npz.exists():
    pack = np.load(validation_npz)
    validation_coords = pack['coords']
    if validation_coords.shape != (N_VALIDATION, 20, 2):
        raise ValueError(f'Unexpected saved validation shape: {validation_coords.shape}')
else:
    rng = np.random.default_rng(VALIDATION_SEED)
    validation_coords = rng.random((N_VALIDATION, 20, 2), dtype=np.float64).astype(np.float32)
    np.savez_compressed(validation_npz, coords=validation_coords, seed=np.int64(VALIDATION_SEED), count=np.int64(N_VALIDATION), n=np.int64(20))

ckpt_files = [p for p in checkpoint_dir.glob('*_step*_act.pt') if parse_suffix(p) in CHECKPOINT_SUFFIXES_TO_CONSIDER]
ckpt_by_suffix = {parse_suffix(p): p for p in ckpt_files}
if not ckpt_by_suffix:
    raise FileNotFoundError(f'No expected checkpoints in {checkpoint_dir}.')
training_hashes = {Path(item['path']).name: item.get('sha256') for item in training_manifest.get('actor_checkpoints', [])}
for suffix, p in sorted(ckpt_by_suffix.items()):
    expected = training_hashes.get(p.name)
    if expected and sha256_file(p) != expected:
        raise RuntimeError(f'Checkpoint SHA-256 mismatch: {p.name}')

validation_rows = []
for suffix, path in sorted(ckpt_by_suffix.items()):
    model = load_actor(path, 'greedy')
    for instance_id, coords in enumerate(validation_coords):
        tour, length, elapsed = predict_one_instance(
            model, coords, sample_count=1,
            inference_seed=VALIDATION_SEED + suffix * 1000 + instance_id)
        validation_rows.append({
            'n': 20, 'instance_id': instance_id, 'checkpoint_suffix': suffix,
            'completed_optimizer_updates': completed_updates_from_suffix(suffix),
            'checkpoint_file': path.name, 'decode_mode': 'greedy',
            'tour_length': length, 'runtime_seconds_single_measurement': elapsed,
            'tour_indices': ','.join(map(str, tour.tolist()))
        })
    del model; torch.cuda.empty_cache()
    print('Validated checkpoint:', path.name)

validation_df = pd.DataFrame(validation_rows)
validation_csv = validation_dir / 'checkpoint_selection_validation_per_instance.csv'
validation_df.to_csv(validation_csv, index=False)
validation_summary = (validation_df.groupby(['checkpoint_suffix','completed_optimizer_updates','checkpoint_file'])
                      .agg(instances=('instance_id','nunique'), mean_validation_length=('tour_length','mean'),
                           sd_validation_length=('tour_length','std')).reset_index()
                      .sort_values(['mean_validation_length','checkpoint_suffix']))
summary_csv = validation_dir / 'checkpoint_selection_validation_summary.csv'
validation_summary.to_csv(summary_csv, index=False)
display(validation_summary)
selected_row = validation_summary.iloc[0]
selected_suffix = int(selected_row['checkpoint_suffix'])
selected_path = ckpt_by_suffix[selected_suffix]
selection = {
    'run_id': RUN_ID, 'source_commit': COMMIT_SHA,
    'selection_rule': 'minimum mean Euclidean tour length on independent n=20 validation set under greedy decoding only',
    'validation_seed': VALIDATION_SEED, 'validation_instances': N_VALIDATION,
    'selected_checkpoint_file': selected_path.name,
    'checkpoint_filename_suffix': selected_suffix,
    'completed_optimizer_updates': completed_updates_from_suffix(selected_suffix),
    'mean_validation_tour_length': float(selected_row['mean_validation_length']),
    'validation_csv_sha256': sha256_file(validation_csv),
}
selection_path = FINAL_ROOT / 'checkpoint_selection.json'
final_test_dir = FINAL_ROOT / 'final_test_instances'
if final_test_dir.exists() and selection_path.exists():
    old_selection = json.loads(selection_path.read_text())
    if old_selection.get('selected_checkpoint_file') != selected_path.name:
        raise RuntimeError(
            'Checkpoint selection changed after final-test artifacts existed. ' 
            'Do not reselect using a test set that may already have been inspected. ' 
            'Preserve this folder and create a new predeclared evaluation ID if a new experiment is justified.'
        )
selection_path.write_text(json.dumps(selection, indent=2))
print('\nSELECTED CHECKPOINT (validation only):')
print(json.dumps(selection, indent=2))

## Step 3 — Generate the fresh final test set only after checkpoint selection

This cell must run only after Step 2 has selected and saved the checkpoint. These coordinates are independent of both the pilot test set (`4202026`) and validation (`7302026`). All methods will use these identical per-size test instances.

In [ ]:
if not selection_path.is_file():
    raise RuntimeError('Run checkpoint selection first. Final test set generation is blocked until selection exists.')
selection_on_disk = json.loads(selection_path.read_text())
if selection_on_disk.get('selected_checkpoint_file') != selected_path.name:
    raise RuntimeError('Selection file and selected checkpoint disagree.')

final_instance_dir = FINAL_ROOT / 'final_test_instances'
final_instance_dir.mkdir(parents=True, exist_ok=True)
FINAL_TEST_COORDS = {}
instance_manifest = {
    'purpose': 'fresh final test; generated after validation-only checkpoint selection',
    'final_test_base_seed': FINAL_TEST_SEED,
    'effective_seed_rule': 'effective_seed = final_test_base_seed + n',
    'instances_per_size': N_FINAL_TEST,
    'distribution': 'independent uniform coordinates in [0,1]^2, float32 storage',
    'city_sizes': {}, 'validation_seed': VALIDATION_SEED,
    'pilot_test_seed': manifest_pre.get('test_seed'),
}
for n in CITY_SIZES:
    size_seed = FINAL_TEST_SEED + n
    p = final_instance_dir / f'final_test_n{n}_seed{size_seed}_count{N_FINAL_TEST}.npz'
    if p.exists():
        pack = np.load(p)
        arr = pack['coords']
        stored_seed = int(pack['seed']) if 'seed' in pack else None
        if arr.shape != (N_FINAL_TEST, n, 2):
            raise ValueError(f'Unexpected final test shape in {p}: {arr.shape}')
        if stored_seed != size_seed:
            raise ValueError(f'Saved test seed mismatch in {p}: expected {size_seed}, found {stored_seed}; preserve files and inspect manually.')
    else:
        rng = np.random.default_rng(size_seed)
        arr = rng.random((N_FINAL_TEST, n, 2), dtype=np.float64).astype(np.float32)
        np.savez_compressed(p, coords=arr, seed=np.int64(size_seed), base_seed=np.int64(FINAL_TEST_SEED), count=np.int64(N_FINAL_TEST), n=np.int64(n))
    FINAL_TEST_COORDS[n] = arr
    instance_manifest['city_sizes'][str(n)] = {
        'shape': list(arr.shape), 'file': p.name, 'effective_seed': int(size_seed), 'sha256': sha256_file(p)
    }

# Guard against accidentally identical coordinate sets.
assert not np.array_equal(FINAL_TEST_COORDS[20], validation_coords), 'Final test and validation arrays unexpectedly match.'
manifest_path = final_instance_dir / 'final_test_instances_manifest.json'
manifest_path.write_text(json.dumps(instance_manifest, indent=2))
print('Fresh final test instances ready:')
print(json.dumps(instance_manifest, indent=2))

## Step 4 — Final test benchmark (quality, all methods on shared instances)

Run after checkpoint selection and test-set creation. This cell reads the selected checkpoint and evaluates it on final test data; it never re-selects the checkpoint. It preserves raw per-instance tours and lengths. Single-call runtimes below are diagnostic only; use Step 6 for repeatable timing summaries.

In [ ]:
selected_path = checkpoint_dir / selection['selected_checkpoint_file']
if not selected_path.is_file():
    raise FileNotFoundError(selected_path)

quality_path = FINAL_ROOT / 'final_test_per_instance.csv'
if quality_path.exists():
    raise RuntimeError(f'Final-test results already exist at {quality_path}; refusing to overwrite them.')
quality_rows = []
for n in CITY_SIZES:
    instances = FINAL_TEST_COORDS[n]
    for instance_id, coords in enumerate(instances):
        aco_seed = FINAL_TEST_SEED + n * 100000 + instance_id
        for method, solver, seed in [
            ('NN_start0', lambda x: nearest_neighbor(x, start=0), aco_seed),
            ('NN_start0_plus_2opt', lambda x: two_opt(x, nearest_neighbor(x, start=0)), aco_seed),
            ('ACO_fixed_parameters', lambda x, s=aco_seed: ant_colony_tsp(x, seed=s), aco_seed),
        ]:
            t0 = time.perf_counter()
            tour = solver(coords)
            length = tour_length_np(coords, tour)
            elapsed = time.perf_counter() - t0
            if sorted(np.asarray(tour, dtype=int).tolist()) != list(range(n)):
                raise AssertionError(f'{method} returned invalid tour for n={n}, instance={instance_id}')
            quality_rows.append({
                'n': n, 'instance_id': instance_id, 'method': method,
                'checkpoint_file': '', 'completed_optimizer_updates': np.nan,
                'decode_mode': '', 'sample_count': 0, 'tour_length': length,
                'runtime_seconds_single_measurement': elapsed, 'seed': aco_seed,
                'tour_indices': ','.join(map(str, np.asarray(tour, dtype=int).tolist()))
            })
    print('Baselines complete for n=', n)

# Model: chosen checkpoint only; greedy and best-of-512 are separate reported configurations.
for mode, selector, sample_count in [('DRL_greedy', 'greedy', 1), ('DRL_best_of_512', 'sampling', SAMPLE_BUDGET)]:
    model = load_actor(selected_path, selector)
    for n in CITY_SIZES:
        for instance_id, coords in enumerate(FINAL_TEST_COORDS[n]):
            inference_seed = FINAL_TEST_SEED + 9_000_000 + n * 100000 + instance_id
            tour, length, elapsed = predict_one_instance(model, coords, sample_count, inference_seed)
            quality_rows.append({
                'n': n, 'instance_id': instance_id, 'method': mode,
                'checkpoint_file': selected_path.name,
                'completed_optimizer_updates': selection['completed_optimizer_updates'],
                'decode_mode': selector, 'sample_count': sample_count,
                'tour_length': length, 'runtime_seconds_single_measurement': elapsed,
                'seed': inference_seed, 'tour_indices': ','.join(map(str, tour.tolist()))
            })
        print('Selected DRL checkpoint complete:', mode, 'n=', n)
    del model; torch.cuda.empty_cache()

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(quality_path, index=False)
expected_methods = {'NN_start0','NN_start0_plus_2opt','ACO_fixed_parameters','DRL_greedy','DRL_best_of_512'}
counts = quality_df.groupby(['n','method'])['instance_id'].nunique()
if not (counts == N_FINAL_TEST).all() or set(quality_df['method'].unique()) != expected_methods:
    raise AssertionError('Incomplete final test results; inspect the per-instance counts.')
print('\nSaved raw final-test data:', quality_path)
display(quality_df.groupby(['n','method']).agg(
    instances=('instance_id','nunique'), mean_tour_length=('tour_length','mean'),
    sd_tour_length=('tour_length','std'), median_tour_length=('tour_length','median')
).reset_index().round(5))

## Step 5 — Integrity checks, descriptive summary and paired uncertainty

Primary outputs are raw tour lengths and paired differences. The best-observed-reference gap is supplementary and must not be described as an exact optimality gap.

In [ ]:
# Recompute the route length from raw coordinates and saved city order for every row.
for row in quality_df.itertuples(index=False):
    coords = FINAL_TEST_COORDS[int(row.n)][int(row.instance_id)]
    tour = np.asarray([int(x) for x in row.tour_indices.split(',')], dtype=np.int64)
    if sorted(tour.tolist()) != list(range(int(row.n))):
        raise AssertionError(f'Invalid permutation: {row.method}, n={row.n}, instance={row.instance_id}')
    measured = tour_length_np(coords, tour)
    if not np.isfinite(row.tour_length) or abs(measured - float(row.tour_length)) > 1e-6:
        raise AssertionError(f'Tour length mismatch: {row.method}, {measured} vs {row.tour_length}')

key_cols = ['n','instance_id','method']
if quality_df.duplicated(key_cols).any():
    raise AssertionError('Duplicate (n, instance, method) rows.')

reference = quality_df.groupby(['n','instance_id'])['tour_length'].min().rename('best_observed_reference')
quality_df = quality_df.join(reference, on=['n','instance_id'])
quality_df['gap_vs_best_observed_percent'] = 100 * (
    quality_df['tour_length'] / quality_df['best_observed_reference'] - 1
)
quality_df['gap_vs_best_observed_percent'] = quality_df['gap_vs_best_observed_percent'].clip(lower=0)
quality_df.to_csv(quality_path, index=False)

summary = (quality_df.groupby(['n','method'])
           .agg(instances=('instance_id','nunique'), mean_tour_length=('tour_length','mean'),
                sd_tour_length=('tour_length','std'), median_tour_length=('tour_length','median'),
                mean_best_observed_gap_percent=('gap_vs_best_observed_percent','mean'),
                sd_best_observed_gap_percent=('gap_vs_best_observed_percent','std'))
           .reset_index())
summary_path = FINAL_ROOT / 'final_test_summary.csv'
summary.to_csv(summary_path, index=False)
display(summary.sort_values(['n','mean_tour_length']).round(5))

# Paired bootstrap CI of mean length difference (DRL minus comparator), resampling instances.
rng = np.random.default_rng(8802026)
paired_rows = []
for n in CITY_SIZES:
    wide = quality_df[quality_df['n'] == n].pivot(index='instance_id', columns='method', values='tour_length').sort_index()
    for drl_name in ['DRL_greedy','DRL_best_of_512']:
        for baseline_name in ['NN_start0','NN_start0_plus_2opt','ACO_fixed_parameters']:
            delta = (wide[drl_name] - wide[baseline_name]).to_numpy(dtype=float)
            boot_means = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
            for b in range(BOOTSTRAP_REPLICATES):
                boot_means[b] = rng.choice(delta, size=len(delta), replace=True).mean()
            lo, hi = np.quantile(boot_means, [0.025, 0.975])
            paired_rows.append({
                'n': n, 'drl_method': drl_name, 'baseline_method': baseline_name,
                'paired_mean_length_difference_drl_minus_baseline': float(delta.mean()),
                'bootstrap_95pct_ci_low': float(lo), 'bootstrap_95pct_ci_high': float(hi),
                'instances': len(delta), 'bootstrap_replicates': BOOTSTRAP_REPLICATES,
                'interpretation': 'negative favors DRL in mean tour length; descriptive paired CI; no multiplicity correction'
            })
paired_df = pd.DataFrame(paired_rows)
paired_path = FINAL_ROOT / 'paired_comparisons_bootstrap.csv'
paired_df.to_csv(paired_path, index=False)
print('PASS: all routes are permutations; saved lengths match recomputation; all 100 instances exist per method.')
print('Saved summary:', summary_path)
print('Saved paired comparisons:', paired_path)
display(paired_df.round(5))

## Step 6 — Repeatable runtime profile (warm-up + repeated timings)

This optional but recommended cell times 10 pre-specified final-test instances per size, with one warm-up and five timed repetitions per method. Model loading is excluded. For DRL, timing includes input conversion/transfer, forward pass, candidate scoring and chosen-tour retrieval; for CPU heuristics, it includes solver execution and route-length calculation. Use these profile outputs for runtime descriptions rather than the single-measurement column in the quality CSV.

In [ ]:
timing_path = FINAL_ROOT / 'runtime_profile_repeated.csv'
if timing_path.exists():
    raise RuntimeError(f'Runtime profile already exists at {timing_path}; refusing to overwrite it.')
timing_rows = []

def timed_cpu_solver(method, coords, seed):
    t0 = time.perf_counter()
    if method == 'NN_start0':
        tour = nearest_neighbor(coords, start=0)
    elif method == 'NN_start0_plus_2opt':
        tour = two_opt(coords, nearest_neighbor(coords, start=0))
    elif method == 'ACO_fixed_parameters':
        tour = ant_colony_tsp(coords, seed=seed)
    else:
        raise ValueError(method)
    length = tour_length_np(coords, tour)
    return length, time.perf_counter() - t0

CPU_METHODS = ['NN_start0','NN_start0_plus_2opt','ACO_fixed_parameters']
DRL_MODES = [('DRL_greedy','greedy',1),('DRL_best_of_512','sampling',SAMPLE_BUDGET)]

# Explicit warm-up per solver / n; warm-up values are not saved.
for n in CITY_SIZES:
    coords0 = FINAL_TEST_COORDS[n][0]
    for method in CPU_METHODS:
        timed_cpu_solver(method, coords0, FINAL_TEST_SEED + 77 + n)
    for _, selector, count in DRL_MODES:
        model = load_actor(selected_path, selector)
        predict_one_instance(model, coords0, count, FINAL_TEST_SEED + 88000 + n)
        del model; torch.cuda.empty_cache()
print('Warm-up completed for all methods and sizes.')

for n in CITY_SIZES:
    profile_ids = list(range(min(TIMING_INSTANCES_PER_SIZE, N_FINAL_TEST)))
    for instance_id in profile_ids:
        coords = FINAL_TEST_COORDS[n][instance_id]
        for method in CPU_METHODS:
            for rep in range(TIMING_REPEATS):
                seed = FINAL_TEST_SEED + 20_000_000 + n * 100000 + instance_id * 100 + rep
                _, elapsed = timed_cpu_solver(method, coords, seed)
                timing_rows.append({'n': n, 'instance_id': instance_id, 'method': method,
                                    'repeat': rep + 1, 'runtime_seconds': elapsed,
                                    'warmups_per_method_and_size': 1})
    for method, selector, count in DRL_MODES:
        model = load_actor(selected_path, selector)
        for instance_id in profile_ids:
            coords = FINAL_TEST_COORDS[n][instance_id]
            for rep in range(TIMING_REPEATS):
                seed = FINAL_TEST_SEED + 30_000_000 + n * 100000 + instance_id * 100 + rep
                _, _, elapsed = predict_one_instance(model, coords, count, seed)
                timing_rows.append({'n': n, 'instance_id': instance_id, 'method': method,
                                    'repeat': rep + 1, 'runtime_seconds': elapsed,
                                    'warmups_per_method_and_size': 1})
        del model; torch.cuda.empty_cache()
    print('Repeated timing completed for n=', n)

timing_df = pd.DataFrame(timing_rows)
timing_df.to_csv(timing_path, index=False)
timing_summary = (timing_df.groupby(['n','method'])
                  .agg(instances=('instance_id','nunique'), measurements=('runtime_seconds','count'),
                       mean_seconds=('runtime_seconds','mean'), sd_seconds=('runtime_seconds','std'),
                       median_seconds=('runtime_seconds','median'))
                  .reset_index())
timing_summary_path = FINAL_ROOT / 'runtime_profile_summary.csv'
timing_summary.to_csv(timing_summary_path, index=False)
print('Saved runtime profile:', timing_path)
display(timing_summary.round(6))

## Step 7 — Freeze final evaluation provenance

This writes a separate final-evaluation manifest and SHA-256 index. It does not modify any pilot files or thesis files.

In [ ]:
runtime_manifest = {
    'evaluation_id': 'TSP_DRL_Independent_Evaluation_v1_1',
    'created_at_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'project': 'Deep reinforcement learning for the Traveling Salesperson Problem',
    'pilot_run_id': RUN_ID,
    'source_repository': REPO_URL,
    'source_commit': COMMIT_SHA,
    'python': sys.version,
    'platform': __import__('platform').platform(),
    'pytorch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'gpu_memory_gib': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3),
    'validation_seed': VALIDATION_SEED,
    'validation_size_n20': N_VALIDATION,
    'selection_rule': selection['selection_rule'],
    'selected_checkpoint_file': selected_path.name,
    'selected_checkpoint_sha256': sha256_file(selected_path),
    'completed_optimizer_updates': selection['completed_optimizer_updates'],
    'final_test_base_seed': FINAL_TEST_SEED,
    'final_test_seed_derivation': 'effective_seed = final_test_base_seed + n',
    'final_test_instances_per_size': N_FINAL_TEST,
    'city_sizes': CITY_SIZES,
    'sample_budget_best_of': SAMPLE_BUDGET,
    'aco': {'ants': ACO_ANTS, 'iterations': ACO_ITERATIONS, 'alpha': ACO_ALPHA,
            'beta': ACO_BETA, 'rho': ACO_RHO, 'q': ACO_Q},
    'timing': {'warmups': 1, 'repeats': TIMING_REPEATS,
               'instances_per_size': TIMING_INSTANCES_PER_SIZE,
               'model_loading_excluded': True,
               'drl_includes_input_transfer_forward_scoring_and_result_retrieval': True,
               'cpu_includes_solver_and_tour_length': True},
    'caveats': [
        'Gap is relative to the best observed method per instance, not exact optimality.',
        'Checkpoint selected on independent validation n=20 greedy mean only.',
        'The selected checkpoint was frozen before final test evaluation.',
        'One training seed only; these results do not characterize across-seed training variance.',
        'Paired bootstrap intervals are descriptive; no multiplicity correction applied.'
    ],
}
manifest_out = FINAL_ROOT / 'final_evaluation_manifest.json'
manifest_out.write_text(json.dumps(runtime_manifest, indent=2))

paths = [
    selection_path, validation_npz, validation_csv, summary_csv,
    manifest_path, quality_path, summary_path, paired_path,
    timing_path, timing_summary_path, manifest_out,
]
artifact_index = []
for p in paths:
    if p.exists():
        artifact_index.append({'file': str(p.relative_to(FINAL_ROOT)), 'bytes': p.stat().st_size,
                               'sha256': sha256_file(p)})
index_out = FINAL_ROOT / 'artifact_index.json'
index_out.write_text(json.dumps(artifact_index, indent=2))
print('FINAL EVALUATION ARTIFACTS SAVED')
print('Directory:', FINAL_ROOT)
print('Selected checkpoint:', selection['selected_checkpoint_file'],
      f"({selection['completed_optimizer_updates']} optimizer updates)")
print('Artifact index:', index_out)
print(json.dumps(artifact_index, indent=2))
print('\nIMPORTANT: Do not edit thesis tables from this run until you review the raw CSV, paired comparisons, runtime protocol, and manifest.')

## Completion checklist

- [ ] Validation set saved with seed `7302026`; checkpoint selection is recorded in `checkpoint_selection.json`.
- [ ] Fresh final-test set saved with seed `8302026`; no pilot test coordinates reused.
- [ ] `final_test_per_instance.csv` contains 100 paired instances per method and size, with valid tours and independently recomputed lengths.
- [ ] `paired_comparisons_bootstrap.csv` and repeated runtime profile are saved.
- [ ] `final_evaluation_manifest.json` and `artifact_index.json` are saved.
- [ ] Only after reviewing the outputs should the final CSVs/manifests be copied into the local PFE Git workspace and the thesis Chapter IV be revised.

**Do not rerun the original v4 training cell for this evaluation.**